In [ ]:
# =====================================================================

# Week 2: Vector search with metadata filters, tenant isolation, and hybrid retrieval

# Qdrant (in-memory) · payload filters · tenant leak test · BM25 · Reciprocal Rank Fusion

# · evaluation by question type

# =====================================================================

!pip -q install qdrant-client sentence-transformers

import math, re

from collections import Counter

import pandas as pd

from qdrant_client import QdrantClient, models

from sentence_transformers import SentenceTransformer

def header(t):

    print(f"\n{'=' * 72}\n{t}\n{'=' * 72}")

model = SentenceTransformer("all-MiniLM-L6-v2")

DIM = len(model.encode("dimension probe"))

# ---------------- corpus: (chunk_id, tenant_id, doc_type, effective_year, text) ----------------

chunks = [

    ("c01", "bank_a", "fee_schedule", 2026, "Overdraft fee: $35 per item, charged when a transaction exceeds the available balance."),

    ("c02", "bank_a", "fee_schedule", 2026, "Premium checking accounts are exempt from overdraft fees."),

    ("c03", "bank_a", "policy",       2026, "A maximum of three overdraft fees can be charged per business day."),

    ("c04", "bank_a", "regulation",   2026, "Customers must opt in before overdraft coverage applies to ATM and one-time debit card transactions (Reg E, 12 CFR 1005.17)."),

    ("c05", "bank_a", "policy",       2026, "Overdraft protection transfers funds from a linked savings account; a $10 transfer fee applies."),

    ("c06", "bank_a", "fee_schedule", 2026, "Domestic wire transfers cost $25; international wires cost $45."),

    ("c07", "bank_a", "fee_schedule", 2026, "Incoming wire transfers are free for all account types."),

    ("c08", "bank_a", "fee_schedule", 2026, "An NSF fee of $35 applies when an item is returned unpaid due to insufficient funds."),

    ("c09", "bank_a", "hours",        2026, "Branches are open 9am to 5pm Monday through Friday and 9am to noon on Saturday."),

    ("c10", "bank_a", "fee_schedule", 2026, "The monthly maintenance fee is $12, waived with a $1,500 minimum daily balance."),

    ("c11", "bank_a", "ach_codes",    2026, "ACH return code R01 means insufficient funds; R02 means the account is closed."),

    ("c12", "bank_a", "fee_schedule", 2026, "Stop payment requests cost $30 per item and remain in effect for six months."),

    ("c13", "bank_a", "ach_codes",    2026, "ACH return code R03 means no account or unable to locate the account."),

    ("c14", "bank_a", "ach_codes",    2026, "ACH return code R04 means the account number is invalid."),

    ("c15", "bank_a", "ach_codes",    2026, "ACH return code R10 means the customer reports the debit as unauthorized."),

    ("c16", "bank_a", "regulation",   2026, "12 CFR 1005.10 governs preauthorized electronic fund transfers."),

    ("c17", "bank_a", "fee_schedule", 2025, "Overdraft fee: $30 per item (2025 fee schedule, superseded)."),

    ("b01", "bank_b", "fee_schedule", 2026, "Student checking accounts pay an overdraft fee of $15 per item."),

    ("b02", "bank_b", "fee_schedule", 2026, "Wire transfers are free for business checking accounts."),

]

# ---------------- Qdrant: collection, payload indexes, upsert ----------------

client = QdrantClient(":memory:")

COLL = "bank_chunks"

client.create_collection(COLL, vectors_config=models.VectorParams(size=DIM, distance=models.Distance.COSINE))

for field, schema in [("tenant_id", models.PayloadSchemaType.KEYWORD),

                      ("effective_year", models.PayloadSchemaType.INTEGER)]:

    try:

        client.create_payload_index(COLL, field_name=field, field_schema=schema)

    except Exception as e:

        print(f"payload index on {field}: not applied in local mode ({type(e).__name__}); required on a real server")

vecs = model.encode([c[4] for c in chunks], normalize_embeddings=True)

client.upsert(COLL, points=[

    models.PointStruct(id=i, vector=v.tolist(),

                       payload={"chunk_id": c[0], "tenant_id": c[1], "doc_type": c[2],

                                "effective_year": c[3], "text": c[4]})

    for i, (c, v) in enumerate(zip(chunks, vecs))

])

print("points stored:", client.count(COLL).count)

def tenant_filter(tenant, current_only=True):

    must = [models.FieldCondition(key="tenant_id", match=models.MatchValue(value=tenant))]

    if current_only:

        must.append(models.FieldCondition(key="effective_year", range=models.Range(gte=2026)))

    return models.Filter(must=must)

def dense_search(query, tenant, k=10, current_only=True):

    qv = model.encode(query, normalize_embeddings=True).tolist()

    res = client.query_points(COLL, query=qv, query_filter=tenant_filter(tenant, current_only), limit=k)

    return [p.payload["chunk_id"] for p in res.points]

def unsafe_dense_search_no_tenant(query, k=5):

    # Demo only: what happens when the tenant filter is forgotten.

    qv = model.encode(query, normalize_embeddings=True).tolist()

    return [p.payload["chunk_id"] for p in client.query_points(COLL, query=qv, limit=k).points]

# ---------------- BM25 ----------------

TOKEN = re.compile(r"[a-z0-9]+(?:\.[a-z0-9]+)*")   # keeps codes like 1005.17 and r04 intact

STOP = {"the", "a", "an", "is", "are", "of", "for", "to", "what", "does", "do", "i", "my", "in",

        "on", "and", "how", "there", "can", "if", "at", "be", "you", "it", "with", "or", "by", "much", "many"}

def stem(t):

    return t[:-1] if len(t) > 3 and t.endswith("s") and not t.endswith("ss") else t

def tokenize(text):

    return [stem(t) for t in TOKEN.findall(text.lower()) if t not in STOP]

class BM25:

    def __init__(self, docs, k1=1.5, b=0.75):

        self.toks = {d: tokenize(t) for d, t in docs.items()}

        self.tf = {d: Counter(t) for d, t in self.toks.items()}

        self.N = len(docs)

        self.avgdl = sum(len(t) for t in self.toks.values()) / self.N

        df = Counter(term for toks in self.toks.values() for term in set(toks))

        self.idf = {t: math.log((self.N - n + 0.5) / (n + 0.5) + 1) for t, n in df.items()}

        self.k1, self.b = k1, b

    def search(self, query, k=10):

        scores = {}

        for d, tf in self.tf.items():

            dl, s = len(self.toks[d]), 0.0

            for t in tokenize(query):

                f = tf.get(t, 0)

                if f:

                    s += self.idf[t] * f * (self.k1 + 1) / (f + self.k1 * (1 - self.b + self.b * dl / self.avgdl))

            if s > 0:

                scores[d] = s

        return sorted(scores, key=scores.get, reverse=True)[:k]

# Pre-filtered: one BM25 index per tenant, current documents only.

bm25 = {t: BM25({c[0]: c[4] for c in chunks if c[1] == t and c[3] >= 2026}) for t in {"bank_a", "bank_b"}}

# ---------------- Reciprocal Rank Fusion ----------------

def rrf_fuse(rankings, k=60, top=10):

    """rankings: list of ranked id lists (best first). Return fused ids, best first."""

    scores = {}

    for ranking in rankings:

        for rank, doc in enumerate(ranking, start=1):

            scores[doc] = scores.get(doc, 0) + 1 / (k + rank)

    return sorted(scores, key=scores.get, reverse=True)[:top]

# Time O(L*n + u log u), space O(u): L lists of length n, u unique ids.

def hybrid_search(query, tenant, k=10):

    return rrf_fuse([dense_search(query, tenant, k=k), bm25[tenant].search(query, k=k)], top=k)

# ---------------- metrics + golden set ----------------

def recall_at_k(ranked, relevant, k):

    return len(set(ranked[:k]) & set(relevant)) / len(relevant)

def reciprocal_rank(ranked, relevant):

    for pos, cid in enumerate(ranked, start=1):

        if cid in relevant:

            return 1 / pos

    return 0.0

golden = [

    ("What happens if I spend more than I have in my account?",          {"c01"}, "paraphrase"),

    ("Can money move automatically from my savings to cover a shortfall?", {"c05"}, "paraphrase"),

    ("How much does it cost to send money abroad by wire?",              {"c06"}, "paraphrase"),

    ("Are you open on weekends?",                                        {"c09"}, "paraphrase"),

    ("What is the fee for a bounced check?",                             {"c08"}, "paraphrase"),

    ("What does 12 CFR 1005.17 require?",                                {"c04"}, "exact_code"),

    ("What is the NSF fee?",                                             {"c08"}, "exact_code"),

    ("What does ACH return code R02 mean?",                              {"c11"}, "exact_code"),

    ("What does R01 stand for?",                                         {"c11"}, "exact_code"),

    ("What does ACH return code R04 mean?",                              {"c14"}, "exact_code"),

    ("What does R10 mean?",                                              {"c15"}, "exact_code"),

    ("What does 12 CFR 1005.10 govern?",                                 {"c16"}, "exact_code"),

    ("Do premium customers pay overdraft charges?",                      {"c02"}, "condition"),

    ("Is there a charge for receiving a wire?",                          {"c07"}, "condition"),

    ("How do I avoid the monthly account fee?",                          {"c10"}, "condition"),

    ("Which accounts don't pay for overdrafts?",                         {"c02"}, "condition"),

    ("How many overdraft fees can I get in one day?",                    {"c03"}, "number"),

    ("How long does a stop payment last?",                               {"c12"}, "number"),

    ("How much is the overdraft fee?",                                   {"c01"}, "number"),

]

def evaluate(name, search_fn):

    rows = []

    for q, rel, qtype in golden:

        ranked = search_fn(q)

        rows.append({"method": name, "type": qtype, "question": q, "expected": ",".join(sorted(rel)),

                     "got_top1": ranked[0] if ranked else None, "top3": ranked[:3],

                     "r@1": recall_at_k(ranked, rel, 1), "r@3": recall_at_k(ranked, rel, 3),

                     "rr": reciprocal_rank(ranked, rel)})

    return rows

# ================= RUN =================

header("0. TOKENIZER CHECK (codes intact, simple stemming)")

print(tokenize("What does 12 CFR 1005.17 require? ACH R04 fees and transfers!"))

header("1. FRESHNESS FILTER: 'How much is the overdraft fee?' (bank_a)")

print("without date filter:", dense_search("How much is the overdraft fee?", "bank_a", k=3, current_only=False))

print("with date filter:   ", dense_search("How much is the overdraft fee?", "bank_a", k=3, current_only=True))

header("2. TENANT ISOLATION")

leak_q = "What overdraft fee do student accounts pay?"

print("NO tenant filter (bug):", unsafe_dense_search_no_tenant(leak_q))

as_a = dense_search(leak_q, "bank_a", k=10)

as_b = dense_search(leak_q, "bank_b", k=10)

print("as bank_a:", as_a)

print("as bank_b:", as_b)

print("LEAK TEST:", "PASS" if not any(c.startswith("b") for c in as_a) and as_b[:1] == ["b01"] else "FAIL")

header("3. RRF CHECK")

print(rrf_fuse([["A", "x", "B"], ["B", "y", "z", "A"]])[:2], "(expect ['B', 'A'])")

print(rrf_fuse([["X", "Y"], ["Z", "W", "X"]])[:2], "(expect ['X', 'Z'])")

header("4. DENSE vs BM25 vs HYBRID (bank_a, current docs, 19 questions; 1 question ≈ 5.3 points)")

methods = [("dense",      lambda q: dense_search(q, "bank_a", k=10)),

           ("bm25",       lambda q: bm25["bank_a"].search(q, k=10)),

           ("hybrid_rrf", lambda q: hybrid_search(q, "bank_a", k=10))]

df = pd.DataFrame([row for name, fn in methods for row in evaluate(name, fn)])

print(df.groupby("method")[["r@1", "r@3", "rr"]].mean().rename(columns={"rr": "MRR"}).round(3).to_string())

header("5. RECALL@1 BY QUESTION TYPE")

print(df.pivot_table(index="type", columns="method", values="r@1", aggfunc="mean").round(2).to_string())

header("6. MISSES AT RANK 1")

misses = df[df["r@1"] == 0][["method", "type", "question", "expected", "got_top1", "top3"]]

print(misses.to_string(index=False) if len(misses) else "No misses.")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

/tmp/ipykernel_785/4109596237.py:89: UserWarning: Payload indexes have no effect in the local Qdrant, except text indexes with `scoring`, which enable text queries. Please use server Qdrant if you need payload indexes.
  client.create_payload_index(COLL, field_name=field, field_schema=schema)


points stored: 19

0. TOKENIZER CHECK (codes intact, simple stemming)
['12', 'cfr', '1005.17', 'require', 'ach', 'r04', 'fee', 'transfer']

1. FRESHNESS FILTER: 'How much is the overdraft fee?' (bank_a)
without date filter: ['c17', 'c01', 'c03']
with date filter:    ['c01', 'c03', 'c05']

2. TENANT ISOLATION
NO tenant filter (bug): ['b01', 'c01', 'c17', 'c02', 'c03']
as bank_a: ['c01', 'c02', 'c03', 'c05', 'c08', 'c04', 'c10', 'c12', 'c07', 'c16']
as bank_b: ['b01', 'b02']
LEAK TEST: PASS

3. RRF CHECK
['B', 'A'] (expect ['B', 'A'])
['X', 'Z'] (expect ['X', 'Z'])

4. DENSE vs BM25 vs HYBRID (bank_a, current docs, 19 questions; 1 question ≈ 5.3 points)
              r@1    r@3    MRR
method                         
bm25        0.789  0.895  0.844
dense       0.737  1.000  0.860
hybrid_rrf  0.737  0.947  0.851

5. RECALL@1 BY QUESTION TYPE
method      bm25  dense  hybrid_rrf
type                               
condition   0.75   0.50        0.50
exact_code  1.00   0.86        0.86
number